# Analytic Hierarchy Process (AHP)

This notebook applies the Analytic Hierarchy Process (AHP) to prioritize projects based on multiple evaluation criteria.

The analysis defines the relative importance of each criterion, evaluates the consistency of the pairwise comparisons, normalizes the project data, and calculates an overall AHP score for each project.

## 1. Configure Project Path

The project root directory is added to the Python path to allow consistent access to modules and project directories.

In [30]:
import sys
from pathlib import Path

project_root = Path().resolve().parent

if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

## 2. Import Libraries

Pandas is used for data manipulation and analysis, while NumPy is used for numerical calculations.

In [31]:
import numpy as np
import pandas as pd

## 3. Load Dataset

The project dataset is loaded from the raw data directory.

The dataset contains the project characteristics used in the portfolio prioritization analysis.

In [32]:
df = pd.read_csv(
    "../data/raw/projects.csv"
)

df.shape

(4000, 51)

## 4. Define Evaluation Criteria

Six criteria are used to evaluate the projects.

Each criterion is classified as either a maximization or minimization criterion according to its desired contribution to project prioritization.

In [33]:
criteria_objectives = {
    "Project_Budget_USD": "min",
    "Estimated_Timeline_Months": "min",
    "Complexity_Score": "min",
    "Previous_Delivery_Success_Rate": "max",
    "Resource_Availability": "max",
    "Historical_Risk_Incidents": "min",
}

criteria_names = list(
    criteria_objectives.keys()
)

criteria_names

['Project_Budget_USD',
 'Estimated_Timeline_Months',
 'Complexity_Score',
 'Previous_Delivery_Success_Rate',
 'Resource_Availability',
 'Historical_Risk_Incidents']

## 5. Define Pairwise Comparison Matrix

The pairwise comparison matrix represents the relative importance of the evaluation criteria.

The comparisons follow the fundamental AHP scale, where values greater than 1 indicate that the row criterion is considered more important than the column criterion, while reciprocal values represent lower relative importance.

In [34]:
pairwise_matrix = pd.DataFrame(
    [
        [1,   1/3, 1/5, 1/7, 1/3, 1/5],
        [3,   1,   1/3, 1/5, 1/2, 1/3],
        [5,   3,   1,   1/3, 3,   1],
        [7,   5,   3,   1,   5,   3],
        [3,   2,   1/3, 1/5, 1,   1/2],
        [5,   3,   1,   1/3, 2,   1],
    ],
    index=criteria_names,
    columns=criteria_names,
)

pairwise_matrix

,Project_Budget_USD,Estimated_Timeline_Months,Complexity_Score,Previous_Delivery_Success_Rate,Resource_Availability,Historical_Risk_Incidents
Project_Budget_USD,1,0.333333,0.200000,0.142857,0.333333,0.200000
Estimated_Timeline_Months,3,1.000000,0.333333,0.200000,0.500000,0.333333
Complexity_Score,5,3.000000,1.000000,0.333333,3.000000,1.000000
Previous_Delivery_Success_Rate,7,5.000000,3.000000,1.000000,5.000000,3.000000
Resource_Availability,3,2.000000,0.333333,0.200000,1.000000,0.500000
Historical_Risk_Incidents,5,3.000000,1.000000,0.333333,2.000000,1.000000


## 6. Normalize Pairwise Comparisons

Each value in the pairwise comparison matrix is divided by the sum of its respective column.

This produces a normalized matrix in which the relative proportions of the criteria can be compared.

In [35]:
normalized_pairwise = pairwise_matrix.div(
    pairwise_matrix.sum(axis=0),
    axis=1
)

normalized_pairwise

,Project_Budget_USD,Estimated_Timeline_Months,Complexity_Score,Previous_Delivery_Success_Rate,Resource_Availability,Historical_Risk_Incidents
Project_Budget_USD,0.041667,0.023256,0.034091,0.064655,0.028169,0.033149
Estimated_Timeline_Months,0.125000,0.069767,0.056818,0.090517,0.042254,0.055249
Complexity_Score,0.208333,0.209302,0.170455,0.150862,0.253521,0.165746
Previous_Delivery_Success_Rate,0.291667,0.348837,0.511364,0.452586,0.422535,0.497238
Resource_Availability,0.125000,0.139535,0.056818,0.090517,0.084507,0.082873
Historical_Risk_Incidents,0.208333,0.209302,0.170455,0.150862,0.169014,0.165746


## 7. Calculate AHP Criteria Weights

The AHP priority vector is calculated by taking the mean of each row of the normalized pairwise comparison matrix.

The resulting values represent the relative weight assigned to each evaluation criterion.

In [36]:
ahp_weights = normalized_pairwise.mean(
    axis=1
)

ahp_weights

Project_Budget_USD                0.037498
Estimated_Timeline_Months         0.073268
Complexity_Score                  0.193037
Previous_Delivery_Success_Rate    0.420704
Resource_Availability             0.096542
Historical_Risk_Incidents         0.178952
dtype: float64

In [37]:
print(
    "Sum of AHP weights:",
    ahp_weights.sum()
)

Sum of AHP weights: 0.9999999999999999


## 8. Analyze Criteria Weights

The criteria weights are sorted to identify which factors have the greatest influence on the final project prioritization.

In [38]:
ahp_weights.sort_values(
    ascending=False
)

Previous_Delivery_Success_Rate    0.420704
Complexity_Score                  0.193037
Historical_Risk_Incidents         0.178952
Resource_Availability             0.096542
Estimated_Timeline_Months         0.073268
Project_Budget_USD                0.037498
dtype: float64

## 9. Evaluate AHP Consistency

The consistency of the pairwise comparisons is evaluated using the Consistency Ratio (CR).

A Consistency Ratio below 0.10 is generally considered acceptable for an AHP pairwise comparison matrix.

In [39]:
weighted_sum = pairwise_matrix.dot(
    ahp_weights
)

consistency_vector = (
    weighted_sum
    / ahp_weights
)

lambda_max = consistency_vector.mean()

n = len(criteria_names)

consistency_index = (
    lambda_max - n
) / (n - 1)

random_index = 1.24

consistency_ratio = (
    consistency_index
    / random_index
)

print(
    f"Lambda max: {lambda_max:.4f}"
)

print(
    f"Consistency Index: {consistency_index:.4f}"
)

print(
    f"Consistency Ratio: {consistency_ratio:.4f}"
)

Lambda max: 6.1758
Consistency Index: 0.0352
Consistency Ratio: 0.0284


In [40]:
if consistency_ratio < 0.10:
    print("The pairwise comparison matrix is consistent.")
else:
    print("The pairwise comparison matrix is not sufficiently consistent.")

The pairwise comparison matrix is consistent.


## 10. Build Decision Matrix

The decision matrix contains the project-level values for the six evaluation criteria.

Each row represents a project and each column represents an evaluation criterion.

In [41]:
decision_matrix = df[
    criteria_names
].copy()

decision_matrix.head()

,Project_Budget_USD,Estimated_Timeline_Months,Complexity_Score,Previous_Delivery_Success_Rate,Resource_Availability,Historical_Risk_Incidents
0,1526276.55,32,9.70,0.80,0.98,2
1,390790.15,9,2.72,0.73,0.95,2
2,246674.76,6,2.04,0.91,0.79,2
3,1427830.63,17,7.54,0.71,0.52,1
4,1696746.64,24,6.68,0.83,0.58,1


## 11. Normalize Decision Matrix

The project criteria are normalized to a common scale between 0 and 1.

For maximization criteria, higher values receive higher scores. For minimization criteria, lower values receive higher scores.

In [42]:
normalized_decision = decision_matrix.copy()

for criterion, objective in criteria_objectives.items():
    minimum = decision_matrix[criterion].min()
    maximum = decision_matrix[criterion].max()

    if maximum == minimum:
        normalized_decision[criterion] = 1

    elif objective == "max":
        normalized_decision[criterion] = (
            decision_matrix[criterion] - minimum
        ) / (maximum - minimum)

    elif objective == "min":
        normalized_decision[criterion] = (
            maximum - decision_matrix[criterion]
        ) / (maximum - minimum)

    else:
        raise ValueError(
            f"Invalid objective: {objective}"
        )

normalized_decision.head()

,Project_Budget_USD,Estimated_Timeline_Months,Complexity_Score,Previous_Delivery_Success_Rate,Resource_Availability,Historical_Risk_Incidents
0,0.621246,0.117647,0.035800,0.773810,0.971429,0.750
1,0.935873,0.794118,0.868735,0.690476,0.928571,0.750
2,0.975805,0.882353,0.949881,0.904762,0.700000,0.750
3,0.648524,0.558824,0.293556,0.666667,0.314286,0.875
4,0.574012,0.352941,0.396181,0.809524,0.400000,0.875


## 12. Calculate Weighted AHP Scores

The normalized decision matrix is multiplied by the AHP criterion weights.

The weighted values are then summed for each project to obtain the overall AHP score.

In [43]:
weighted_matrix = normalized_decision.mul(
    ahp_weights,
    axis=1
)

ahp_scores = weighted_matrix.sum(
    axis=1
)

ahp_scores.head()

0    0.592368
1    0.775320
2    0.867031
3    0.589323
4    0.659631
dtype: float64

## 13. Generate AHP Ranking

Projects are ranked according to their AHP scores.

Higher scores indicate projects with more favorable characteristics according to the selected criteria and their AHP weights.

In [44]:
ahp_ranking = df[
    ["Project_ID"]
].copy()

ahp_ranking["AHP_Score"] = (
    ahp_scores
)

ahp_ranking = ahp_ranking.sort_values(
    by="AHP_Score",
    ascending=False
).reset_index(drop=True)

ahp_ranking.head(10)

,Project_ID,AHP_Score
0,PROJ_0614,0.948052
1,PROJ_3497,0.937929
2,PROJ_1686,0.935575
3,PROJ_0940,0.932549
4,PROJ_2031,0.930085
5,PROJ_3982,0.920823
6,PROJ_2845,0.918382
7,PROJ_1070,0.914523
8,PROJ_2119,0.913478
9,PROJ_2068,0.913064


## 14. Validate AHP Ranking

The ranking is validated to confirm that all projects received a score and a unique ranking position.

In [45]:
ahp_ranking["AHP_Rank"] = (
    ahp_ranking.index + 1
)

print(
    "Number of projects:",
    len(ahp_ranking)
)

print(
    "Number of unique projects:",
    ahp_ranking["Project_ID"].nunique()
)

print(
    "Number of unique ranks:",
    ahp_ranking["AHP_Rank"].nunique()
)

print(
    "Minimum AHP score:",
    ahp_ranking["AHP_Score"].min()
)

print(
    "Maximum AHP score:",
    ahp_ranking["AHP_Score"].max()
)

Number of projects: 4000
Number of unique projects: 4000
Number of unique ranks: 4000
Minimum AHP score: 0.2509792325065746
Maximum AHP score: 0.9480517480668883


## 15. Save AHP Results

The AHP ranking is saved as a processed dataset so that it can be reused by subsequent analyses without recalculating the complete AHP model.

In [46]:
ahp_ranking.to_csv(
    "../data/processed/ahp_ranking.csv",
    index=False
)

print(
    "AHP ranking saved successfully."
)

AHP ranking saved successfully.


## 16. Inspect Top-Priority Projects

The highest-ranked projects are displayed to provide an initial view of the portfolio priorities generated by the AHP method.

In [47]:
ahp_ranking.head(20)

,Project_ID,AHP_Score,AHP_Rank
0,PROJ_0614,0.948052,1
1,PROJ_3497,0.937929,2
2,PROJ_1686,0.935575,3
3,PROJ_0940,0.932549,4
4,PROJ_2031,0.930085,5
5,PROJ_3982,0.920823,6
6,PROJ_2845,0.918382,7
7,PROJ_1070,0.914523,8
8,PROJ_2119,0.913478,9
9,PROJ_2068,0.913064,10
